In [1]:
from pymodbus.client import ModbusTcpClient
import time

class ModbusTCP:
    """
    A helper class for communicating with Modbus TCP devices.
    Supports reading/writing coils, inputs, and registers.
    Now supports both decimal and hexadecimal addressing.
    """

    def __init__(self, host='192.168.1.100', port=502):
        self.client = ModbusTcpClient(host=host, port=port)
        self.host = host
        self.port = port

    # ---------------------------------------------------------
    # UTIL FUNCTIONS
    # ---------------------------------------------------------

    def _parse_address(self, addr):
        """
        Parse address from any numeric base to decimal.

        Supported formats:
            - Decimal: "10", "123"
            - Hex: "0x10", "FF", "1A"
            - Octal: "0o77", "077"
            - Binary: "0b1010"
        """
        if isinstance(addr, int):
            return addr

        if not isinstance(addr, str):
            raise ValueError(f"Unsupported address type: {type(addr)}")

        s = addr.strip().lower()

        # Prefix formats
        if s.startswith("0x"):
            return int(s, 16)
        if s.startswith("0b"):
            return int(s, 2)
        if s.startswith("0o"):
            return int(s, 8)

        # Pure digits → decimal
        if s.isdigit():
            return int(s, 10)

        # Hex without prefix (A–F)
        hex_chars = set("0123456789abcdef")
        if all(c in hex_chars for c in s):
            return int(s, 16)

        raise ValueError(f"Cannot auto-detect numeric base from: {addr}")

    # ---------------------------------------------------------
    # CONNECTION
    # ---------------------------------------------------------

    def connect(self):
        self.client.connect()
        return self.client.connected

    def disconnect(self):
        self.client.close()
        return not self.client.connected

    # ---------------------------------------------------------
    # DIGITAL OUTPUT (COILS)
    # ---------------------------------------------------------

    def read_status_output(self, address):
        address = self._parse_address(address)
        response = self.client.read_coils(address=address, count=1)
        if response.isError():
            print(f"Error reading digital coil at address {address}")
            return None
        return response.bits[0]

    def digital_write(self, address, value):
        address = self._parse_address(address)
        response = self.client.write_coil(address=address, value=value)
        if response.isError():
            print(f"Error writing digital coil at address {address}")
            return None
        return True

    def multiple_digital_write(self, address, values=[0, 0, 0, 0]):
        address = self._parse_address(address)
        response = self.client.write_coils(address=address, values=values)
        if response.isError():
            print(f"Error writing multiple digital coils at address {address}")
            return None
        return True

    # ---------------------------------------------------------
    # DIGITAL INPUT
    # ---------------------------------------------------------

    def digital_input(self, address, count=1):
        address = self._parse_address(address)
        response = self.client.read_discrete_inputs(address=address, count=count)
        if response.isError():
            print(f"Error reading digital input at address {address}")
            return None
        return response.bits[0] if count == 1 else response.bits

    # ---------------------------------------------------------
    # ANALOG INPUT
    # ---------------------------------------------------------

    def analog_read(self, address, count=1):
        address = self._parse_address(address)
        response = self.client.read_input_registers(address=address, count=count)
        if response.isError():
            print(f"Error reading analog input at address {address}")
            return None
        return response.registers[0] if count == 1 else response.registers

    # ---------------------------------------------------------
    # HOLDING REGISTERS
    # ---------------------------------------------------------

    def read_holding_registers(self, address, count=1, slave_id=1):
        address = self._parse_address(address)
        response = self.client.read_holding_registers(
            address=address, count=count, unit=slave_id
        )
        if response.isError():
            print(f"Error reading holding registers at address {address}")
            return None
        return response.registers

    def write_holding_register(self, address, value):
        address = self._parse_address(address)
        response = self.client.write_register(address=address, value=value)
        if response.isError():
            print(f"Error writing holding register at address {address}")
            return None
        return True

    def multiple_write_holding_registers(self, address, values=[0, 0]):
        address = self._parse_address(address)
        response = self.client.write_registers(address=address, values=values)
        if response.isError():
            print(f"Error writing multiple holding registers at address {address}")
            return None
        return True


# =====================================================================
# TEST BLOCK
# =====================================================================

if __name__ == "__main__":
    modbus = ModbusTCP(host="192.168.1.100")

    if not modbus.connect():
        print("Cannot connect to Modbus device!")
    else:
        print("Connected.")
        print("===== START MODBUS AUTO TEST =====")

    # Address Ranges
    COIL_START = "0x4000"
    COIL_END   = "0x4005"

    REG_START = "0x0400"
    REG_END   = "0x0403"

    INPUT_START = "0x0000"
    INPUT_END   = "0x0003"

    ANALOG_START = "0x0001"
    ANALOG_END   = "0x0004"

    DELAY = 0.5

    # ---------------------------------------------------------
    # 1) TEST COILS
    # ---------------------------------------------------------
    print("\n=== TEST COILS (ON → OFF) ===")

    for addr_hex in range(int(COIL_START, 16), int(COIL_END, 16) + 1):
        modbus.digital_write(addr_hex, 1)
        value = modbus.read_status_output(addr_hex)
        print(f"COIL ON : Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    for addr_hex in range(int(COIL_START, 16), int(COIL_END, 16) + 1):
        modbus.digital_write(addr_hex, 0)
        value = modbus.read_status_output(addr_hex)
        print(f"COIL OFF: Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    # ---------------------------------------------------------
    # 2) TEST HOLDING REGISTERS
    # ---------------------------------------------------------
    print("\n=== TEST HOLDING REGISTERS ===")

    MAX_VAL = 27647

    for addr_hex in range(int(REG_START, 16), int(REG_END, 16) + 1):
        modbus.write_holding_register(addr_hex, MAX_VAL)
        value = modbus.read_holding_registers(addr_hex)[0]
        print(f"WRITE MAX: Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    for addr_hex in range(int(REG_START, 16), int(REG_END, 16) + 1):
        modbus.write_holding_register(addr_hex, 0)
        value = modbus.read_holding_registers(addr_hex)[0]
        print(f"WRITE ZERO: Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    # ---------------------------------------------------------
    # 3) TEST DIGITAL INPUT
    # ---------------------------------------------------------
    print("\n=== TEST DIGITAL INPUT ===")

    for addr_hex in range(int(INPUT_START, 16), int(INPUT_END, 16) + 1):
        value = modbus.digital_input(addr_hex)
        print(f"INPUT: Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    # ---------------------------------------------------------
    # 4) TEST ANALOG INPUT
    # ---------------------------------------------------------
    print("\n=== TEST ANALOG INPUT ===")

    for addr_hex in range(int(ANALOG_START, 16), int(ANALOG_END, 16) + 1):
        value = modbus.analog_read(addr_hex)
        print(f"ANALOG: Dec={addr_hex} Hex={hex(addr_hex)} Value={value}")
        time.sleep(DELAY)

    print("\n===== END MODBUS TEST =====")

    modbus.disconnect()
    print("Disconnected.")


Connected.
===== START MODBUS AUTO TEST =====

=== TEST COILS (ON → OFF) ===
COIL ON : Dec=16384 Hex=0x4000 Value=True
COIL ON : Dec=16385 Hex=0x4001 Value=True
COIL ON : Dec=16386 Hex=0x4002 Value=True
COIL ON : Dec=16387 Hex=0x4003 Value=True
COIL ON : Dec=16388 Hex=0x4004 Value=True
COIL ON : Dec=16389 Hex=0x4005 Value=True
COIL OFF: Dec=16384 Hex=0x4000 Value=False
COIL OFF: Dec=16385 Hex=0x4001 Value=False
COIL OFF: Dec=16386 Hex=0x4002 Value=False
COIL OFF: Dec=16387 Hex=0x4003 Value=False
COIL OFF: Dec=16388 Hex=0x4004 Value=False
COIL OFF: Dec=16389 Hex=0x4005 Value=False

=== TEST HOLDING REGISTERS ===


TypeError: ModbusClientMixin.read_holding_registers() got an unexpected keyword argument 'unit'